In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from json import load as load_json
import re
from datetime import datetime, date

In [ ]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

In [ ]:
with open("../../data/raw/players_url.txt", "r") as f:
    players_url = f.read().splitlines()
players_url[:5]

columns = [
        "ID",
        "Name",
        "From",
        "To",
        "Height",
        "Weight",
        "Birth Date",
        "Age",
        "Nationality",
        "Position",
        "Shoots",
        "College",
        "Experience"
]
try:
    players_data = pd.read_csv("../../data/raw/players_data.csv")
except:
    players_data = pd.DataFrame(columns = columns)
players_data.head()

In [ ]:
def get_date(month_and_day: str, year: str):
    """
    Convert month/day string and year to a date object.
    
    Args:
        month_and_day: e.g., "April 7"
        year: e.g., "1999"
    
    Returns:
        datetime.date object
    """
    date_string = f"{month_and_day}, {year}"
    return datetime.strptime(date_string, "%B %d, %Y").date()

def player_info(url: str, headers: dict):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = pd.Series()
        
        while (True):
                try:
                        info_tag = [div for div in soup.select("#meta > div") if (div.get("class") == None or "nothumb" in div.get("class"))][0]
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        per_game_stats_tag = soup.select("#per_game_stats > tbody")[0]
                
        # `Name`
        try:
                name = info_tag.select(" h1 > span")[0].text
                result["Name"] = name
        except:
                pass
        
        # `From` and `TO` and "Experience"
        try:
                seasons = per_game_stats_tag.select("tr")
                fok = 0
                prv_year = ""
                exp = 0
                for season in seasons:
                        try:
                                comp = season.find(attrs={"data-stat": "comp_name_abbr"}).a.text.strip()
                        except:
                                comp = ""
                        # We are only looking for NBA seasons
                        if (comp != "NBA"):
                                continue
                        try:
                                year = season.find(attrs={"data-stat": "year_id"}).a.text.strip()
                        except:
                                try:
                                        year = season.th.text
                                except:
                                        continue
                        
                        if fok == 0:
                                result["From"] = year
                                fok = 1

                        result["To"] = year
                        if (year != prv_year):
                                exp += 1
                        prv_year = year
                result["Experience"] = exp
        except:
                pass
        
        # `Height` and `Height` 
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Position' in text))
                if strong_tag:
                        pt = strong_tag.find_parent('p')
                pt = pt.next_sibling
                while pt and pt.name is None:
                        pt = pt.next_sibling
                
                txt = pt.text
                # txt = info_tag.select("p:nth-child(6)")[0].text + soup.select("#meta > div > p:nth-child(7)")[0].text
                
                cm_match = re.search(r'(\d+)cm', txt)
                kg_match = re.search(r'(\d+)kg', txt)
                cm = float(cm_match.group(1)) if cm_match else None
                kg = float(kg_match.group(1)) if kg_match else None
                
                result["Height"] = cm
                result["Weight"] = kg
        except:
                pass
        
        # `Birth Date` and `Age`
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Born' in text))
                if strong_tag:
                        pt = strong_tag.find_parent('p')
                
                d = pt.select("#necro-birth")[0].get("data-birth")
                bd = datetime.strptime(d, "%Y-%m-%d").date()
                result["Birth Date"] = d
                
                today = date.today()
                age = today.year - bd.year
                
                if (today.month, today.day) < (bd.month, bd.day):
                        age -= 1
                result["Age"] = age
                
        except:
                pass
        
        # `Nationality`
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Born' in text))
                if strong_tag:
                        pt = strong_tag.find_parent('p')
                txt = pt.select("span:nth-child(3)")[0].a.text
                result["Nationality"] = txt
                # go for span:nth-child(4) if you want the country. 
                # [<span class="f-i f-us" style="">us</span>]
                # [<span class="f-i f-eg" style="">eg</span>]
        except:
                pass
        
        # `Position` and `Shoots`
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Position' in text))
                if strong_tag:
                        pt = strong_tag.find_parent('p')
                
                lst = pt.select("strong")
                for strong in lst:
                        if 'Position:' in strong.text:
                                next_sibling = strong.next_sibling
                                if next_sibling:
                                        position = next_sibling.strip().split("\n")[0]
                        elif 'Shoots:' in strong.text:
                                next_sibling = strong.next_sibling
                                if next_sibling:
                                        shoots = next_sibling.strip()
                result["Position"] = position
                result["Shoots"] = shoots
        except:
                pass
        
        # `College`
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('College' in text.strip()))
                if strong_tag:
                        pt = strong_tag.find_parent('p')
                
                txt = pt.a.text
                result["College"] = txt
        except:
                pass
        
        time.sleep(_Delay)
        return result

In [ ]:
for i in tqdm(range(len(players_url))):
        if (i < len(players_data) and ~np.isnan(players_data.loc[i, 'Height'])):
                continue
        result = player_info(players_url[i], headers)
        result["ID"] = i
        players_data.loc[i] = result
        players_data['Birth Date'] = pd.to_datetime(players_data['Birth Date'])
        players_data.to_csv("../../data/raw/players_data.csv", index = False)
        
players_data